# Download các thư viện cần thiết


In [2]:
import os
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Tạo hàm để đọc file parquet (đọc các file parquet - data sau khi được processed)

In [3]:
def read_parquet(train_path: str):
    # Lấy tất cả các file parquet trong thư mục
    files = [os.path.join(train_path, f) for f in os.listdir(train_path) if f.endswith('.parquet')]
    
    # Phân loại các file theo loại tên
    user_chunk_files = [file for file in files if 'user_chunk' in file]
        
    # Đọc các file riêng biệt thành DataFrame
    user_chunk_df = pl.concat([pl.read_parquet(file) for file in user_chunk_files]) if user_chunk_files else None
        
    # Trả về một dictionary chứa các DataFrame
    return user_chunk_df

In [4]:
def read_parquet_item(train_path: str):
    # Lấy tất cả các file parquet trong thư mục
    files = [os.path.join(train_path, f) for f in os.listdir(train_path) if f.endswith('.parquet')]
    
    # Phân loại các file theo loại tên
    item_chunk_files = [file for file in files if 'item_chunk' in file]
        
    # Đọc các file riêng biệt thành DataFrame
    item_chunk_df = pl.concat([pl.read_parquet(file) for file in item_chunk_files]) if item_chunk_files else None
        
    # Trả về một dictionary chứa các DataFrame
    return item_chunk_df

In [5]:
def read_parquet_purchase(train_path: str):
    # Lấy tất cả các file parquet trong thư mục
    files = [os.path.join(train_path, f) for f in os.listdir(train_path) if f.endswith('.parquet')]
    
    # Phân loại các file theo loại tên
    purchase_chunk_files = [file for file in files if 'purchase_history_daily_chunk' in file]
        
    # Đọc các file riêng biệt thành DataFrame
    purchase_chunk_df = pl.concat([pl.read_parquet(file) for file in purchase_chunk_files]) if purchase_chunk_files else None
        
    # Trả về một dictionary chứa các DataFrame
    return purchase_chunk_df

# Tạo hàm lưu file parquet sau mỗi task

In [6]:
def split_and_save_parquet(df, num_files, output_dir):
    """
    Tách DataFrame thành nhiều file Parquet và lưu vào thư mục đích.
    
    :param df: DataFrame cần tách
    :param num_files: Số lượng file Parquet muốn tách
    :param output_dir: Thư mục lưu các file Parquet
    """
    # Đảm bảo thư mục tồn tại
    os.makedirs(output_dir, exist_ok=True)
    
    # Tính số dòng mỗi file sẽ có
    num_rows = df.height
    rows_per_file = num_rows // num_files

    # Tách DataFrame thành các phần và lưu mỗi phần vào một file Parquet
    for i in range(num_files):
        start_row = i * rows_per_file
        # Đảm bảo phần cuối cùng sẽ chứa tất cả các dòng còn lại
        end_row = (i + 1) * rows_per_file if i < num_files - 1 else num_rows
        
        # Tách phần DataFrame
        split_df = df[start_row:end_row]
        
        # Lưu phần DataFrame vào file .parquet
        file_path = os.path.join(output_dir, f"sale_pers.purchase_history_daily_chunk_{i}.parquet")
        split_df.write_parquet(file_path)
        print(f"Đã lưu file: {file_path}")

# Load các dataframe cần thiết

In [7]:
purchase_df = read_parquet_purchase("./preprocessed-dataset")
purchase_df

item_id,quantity,customer_id,created_date,location,price,log_price,discount_rate,channel,payment_bucket,time_between_purchases,month,seasonal_trend,product_engagement_level
str,i32,i32,datetime[μs],i32,f64,f64,f64,str,str,duration[μs],i8,str,str
"""0020120000014""",1,4298411,2024-04-18 09:12:54.587,435,92000.0,11.429555,0.2,"""In-Store""","""cash""",325d 6h 50m 47s 303ms,4,"""Spring""","""High"""
"""6766000000003""",1,1667373,2024-05-06 18:32:27.847,730,255000.0,12.449023,0.105263,"""In-Store""","""cash""",276d 9m 14s 926ms,5,"""Spring""","""High"""
"""2793000000003""",1,4993265,2024-05-06 08:38:13.453,750,62000.0,11.034906,0.0,"""iOS""","""cash""",301d 22h 11m 51s 683ms,5,"""Spring""","""High"""
"""0007140000001""",1,4021166,2024-05-05 21:55:56.197,268,128000.0,11.759793,0.0,"""In-Store""","""qr""",334d 20h 5m 2s 537ms,5,"""Spring""","""High"""
"""0029130000030""",1,3287714,2024-05-05 20:22:17.403,352,69000.0,11.141876,0.0,"""In-Store""","""cash""",343d 1h 20m 43s 270ms,5,"""Spring""","""High"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""6064000000004""",1,6089361,2024-10-27 18:33:00.130,895,37800.0,10.540091,0.0,"""In-Store""","""cash""",132d 7h 41m 29s 600ms,10,"""Autumn""","""High"""
"""2803000000011""",2,6680814,2024-10-27 19:50:26.860,443,41500.0,10.633473,0.153061,"""Android""","""wallet""",261d 22h 36m 310ms,10,"""Autumn""","""High"""
"""5029000000010""",1,2371171,2024-10-27 16:09:58.660,947,65000.0,11.082158,0.057971,"""In-Store""","""qr""",162d 5h 54m 32s 743ms,10,"""Autumn""","""High"""


In [10]:
item_df = read_parquet_item("./preprocessed-dataset")
item_df.head()

item_id,price,category_l1,category,brand_final,target_user_group_final,item_type_final,color_final,origin_final,material_final,sale_status,description_merge,age_bucket_final,price_segment
str,"decimal[38,4]",str,str,str,str,str,str,str,str,i32,str,str,str
"""0502020000004""",99000.0000,"""Babycare""","""Núm ty Dr Brown""","""Dr.Brown's""","""Sơ sinh""",null,"""Đen""","""Ý""","""Silicone""",0,"""Chi tiết sản phẩm …","""1-3M""","""Mid"""
"""0010290040150""",69000.0000,"""Thời trang""","""Bộ quần áo bé gái""","""Con Cưng""","""Bé Gái""","""Bộ quần áo""",null,null,null,0,null,"""2-4Y""","""Mid"""
"""0008010000015""",45000.0000,"""Đồ chơi & Sách""","""Gặm nướu khác""","""Thương hiệu khác""","""Bé Trai""",null,"""Hồng""","""Đức""","""Silicone""",0,"""Chi tiết sản phẩm …",null,"""Low"""
"""0020010000094""",401000.0000,"""Tã""","""Merries_Sơ Sinh""","""Merries Nhật""","""Sơ sinh""",null,"""Đen""","""Nhật Bản, Nhật Bản""","""Giấy, bột giấy, vải không dệt,…",0,"""﻿﻿Tã dán Merries size S 82 miế…","""3-6M""","""High"""
"""0020010000098""",401000.0000,"""Tã""","""Merries_Tã Quần""","""Merries Nhật""","""Sơ sinh""",null,"""Đen""","""Nhật Bản, Nhật Bản""","""Giấy, bột giấy, vải không dệt,…",0,"""﻿﻿﻿Bỉm tã quần Merries size M …","""6-9M""","""High"""


In [8]:
user_df = read_parquet("./preprocessed-dataset")
user_df.head()

customer_id,gender,location,province,membership,region,location_name,install_app,district
i32,str,i32,str,str,str,str,str,str
8220125,"""Nữ""",240,"""Tiền Giang""","""Gold""","""Đồng bằng sông Cửu Long""","""TGI - 364-365 Nguyễn Huệ""","""In-Store""","""Gò Công"""
8220124,"""Nữ""",996,"""Đồng Nai""","""Standard""","""Đông Nam Bộ""","""DON - 569 Quốc lộ 20""","""In-Store""","""Tân Phú"""
8220138,"""Nữ""",606,"""Hồ Chí Minh""","""Standard""","""Đông Nam Bộ""","""HCM - 385 Bùi Đình Túy""","""In-Store""","""Bình Thạnh"""
8220127,"""Nữ""",746,"""Hà Nội""","""Standard""","""Đồng bằng sông Hồng""","""HNI - 16B-4 Nguyễn Văn Lộc""","""In-Store""","""Hà Đông"""
8220130,"""Nam""",418,"""Hồ Chí Minh""","""Standard""","""Đông Nam Bộ""","""HCM - 1069 Tỉnh Lộ 43""","""In-Store""","""Thủ Đức"""


# Task A: Hãy thống kê những sản phẩm hay mua chung và số lần mua chung: item 1 | item 2 | #cooc

Chuyển dữ liệu của trường `created_date` sang dạng `timestamp` để việc xử lý trở nên dễ dàng hơn 

In [11]:
import polars as pl
from itertools import combinations
from collections import Counter

# --- Giả sử df_purchase có cột: customer_id, invoice_id, item_id ---
# df_purchase = pl.read_parquet("purchases.parquet")

# 1️⃣ Gom các sản phẩm trong cùng một hóa đơn
df_grouped = (
    purchase_df.group_by(["customer_id", "created_date"])
    .agg(pl.col("item_id").unique().alias("items"))
)

# 2️⃣ Đếm số lần xuất hiện cặp sản phẩm
cooc_counter = Counter()
for items in df_grouped["items"]:
    if len(items) > 1:
        for pair in combinations(sorted(items), 2):
            cooc_counter[pair] += 1

# 3️⃣ Kết quả
cooc_count = pl.DataFrame({
    "item_1": [i1 for (i1, i2) in cooc_counter.keys()],
    "item_2": [i2 for (i1, i2) in cooc_counter.keys()],
    "cooc_count": list(cooc_counter.values())
}).sort("cooc_count", descending=True)

print(cooc_count.head(10))

shape: (10, 3)
┌───────────────┬───────────────┬────────────┐
│ item_1        ┆ item_2        ┆ cooc_count │
│ ---           ┆ ---           ┆ ---        │
│ str           ┆ str           ┆ i64        │
╞═══════════════╪═══════════════╪════════════╡
│ 2803000000011 ┆ 2803000000013 ┆ 78235      │
│ 2803000000012 ┆ 2803000000013 ┆ 53478      │
│ 2803000000011 ┆ 2803000000012 ┆ 52843      │
│ 2803000000010 ┆ 2803000000012 ┆ 34204      │
│ 2803000000010 ┆ 2803000000013 ┆ 32656      │
│ 1371000000001 ┆ 1371000000002 ┆ 29677      │
│ 3880000000001 ┆ 3880000000002 ┆ 27793      │
│ 2803000000010 ┆ 2803000000011 ┆ 26985      │
│ 1371000000003 ┆ 1371000000006 ┆ 26776      │
│ 0029250010001 ┆ 0029250010003 ┆ 26509      │
└───────────────┴───────────────┴────────────┘


In [12]:
cooc_count.write_parquet("./data/history-chunk/cooc_count.parquet")

In [13]:
pairs = (
    pl.concat([
        cooc_count.select(
            pl.col("item_1").alias("item_id"),
            pl.col("item_2").alias("co_item"),
            pl.col("cooc_count")
        ),
        cooc_count.select(
            pl.col("item_2").alias("item_id"),
            pl.col("item_1").alias("co_item"),
            pl.col("cooc_count")
        )
    ])
)

# 🧠 Lấy top 5 co_item có cooc_count cao nhất cho từng item_id
top_co_items = (
    pairs.sort(["item_id", "cooc_count"], descending=[False, True])
         .group_by("item_id")
         .agg(pl.col("co_item").head(10).alias("top10_co_items"))
)

# 🔗 Gộp vào item_df
item_df = item_df.join(top_co_items, on="item_id", how="left")

# ✅ Kết quả
print(item_df.select(["item_id", "top10_co_items"]).head())


shape: (5, 2)
┌───────────────┬─────────────────────────────────┐
│ item_id       ┆ top10_co_items                  │
│ ---           ┆ ---                             │
│ str           ┆ list[str]                       │
╞═══════════════╪═════════════════════════════════╡
│ 0502020000004 ┆ ["0502020340085", "00071500001… │
│ 0010290040150 ┆ null                            │
│ 0008010000015 ┆ ["0008170000235", "22630000000… │
│ 0020010000094 ┆ ["5950000000001", "00200200001… │
│ 0020010000098 ┆ ["1512000000004", "00200200001… │
└───────────────┴─────────────────────────────────┘


Kiểm tra một vài số liệu của feature mới

In [14]:
target_id = "0502020000004"

top10_list = (
    item_df
    .filter(pl.col("item_id") == target_id)
    .select("top10_co_items")
    .item()   # chuyển giá trị trong cột thành object Python
)

print(top10_list)

shape: (10,)
Series: '' [str]
[
	"0502020340085"
	"0007150000143"
	"0007090000357"
	"0020020000264"
	"2752000000003"
	"2766000000005"
	"0006040000432"
	"6548000000002"
	"4548000000003"
	"4553000000003"
]


In [21]:
target_id = "0502020000004"

# 1️⃣ Lấy danh sách top 10 sản phẩm mua chung
top10_list = (
    item_df
    .filter(pl.col("item_id") == target_id)
    .select("top10_co_items")
    .item()  # chuyển list từ cột sang object Python
)

# 2️⃣ Lọc item_df để lấy thông tin chi tiết của các sản phẩm này
top10_info = (
    item_df
    .filter(pl.col("item_id").is_in(top10_list))
    .select(["item_id", "description_merge"])
)

# 3️⃣ In kết quả
print(top10_info["description_merge"])

shape: (10,)
Series: 'description_merge' [str]
[
	"Chi tiết sản phẩm             …
	"Chi tiết sản phẩm             …
	"﻿﻿﻿﻿﻿﻿Lay  Khoai Tây Tảo Nori…
	"Chi tiết sản phẩmTên sản phẩm:…
	"Chi tiết sản phẩm             …
	"Dầu Tràm Cung Đình không mang …
	null
	"Chi tiết sản phẩm             …
	"Bánh Ăn Sáng C'est Bon Orion S…
	"﻿Được làm từ gạo Japonica, bán…
]


/tmp/ipykernel_2199790/3986287780.py:14: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  .filter(pl.col("item_id").is_in(top10_list))


In [26]:
# Lấy dữ liệu từ cột item_id và description_merge
item_ids = top10_info["item_id"].to_list()
descriptions = top10_info["description_merge"].to_list()

# In kết quả song song với item_id và description_merge
for item_id, description in zip(item_ids, descriptions):
    print(f"Item ID: {item_id}, Description: {description}\n")


Item ID: 0502020340085, Description: Chi tiết sản phẩm                     Tên sản phẩm: Núm ty bình sữa cổ hẹp Dr Brown's 2 cái, số 1, 0-3M         Xuất xứ: USA         Nhà sản xuất: Công ty Handi-Craft Company         Chất liệu sản phẩm: Silicon                     Núm ty bình sữa cổ hẹp Dr Brown's 2 cái, số 1, 0-3M là núm ty dành cho bé sơ sinh của thương hiệu Dr Brown's đến từ Mỹ. Sản phẩm được làm từ silicon y tế an toàn, giúp mang đến cảm giác dễ chịu cho bé như bú mẹ trực tiếp.                     Chất liệu an toàn: Silicon y tế đảm bảo sức khỏe cho bé           Núm ty Dr Brown's được làm từ silicon y tế không BPA, theo tiêu chuẩn an toàn sức khỏe của Mỹ, giúp mẹ hoàn toàn yên tâm khi cho bé sử dụng sản phẩm.            Thiết kế thông minh: Giảm tình trạng đầy hơi, sặc sữa           Núm ty được thiết kế với lỗ tiết sữa thông minh giúp sữa chảy vừa phải, hạn chế tình trạng đầy hơi và sặc sữa cho trẻ trong quá trình bú.            Nhiều kích cỡ lựa chọn: Đáp ứng nhu cầu từng giai 

# Task B. Hãy dự đoán tuổi của em bé dựa trên:
- Thông tin "age_group" của bảng item: Ngày đầu tiên mua

- Thông tin sữa có chữ "Step 1": Ngày đầu tiên mua

- Thông tin sữa có chữ "Mom": Ngày cuối cùng mua

Tạo bảng dự đoán: | customer_id | first_date_buy_step 1 | age_by_step1 | first_date_buy_age_group_0-3M | age_by_age_group | last_date_buy_milk4mom | age_by_milk4mom.

In [80]:
# 1️⃣ Lấy danh sách item_id
step1_items = item_df.filter(
    pl.col("description_merge").str.contains("(?i)Step 1") 
)["item_id"]
print(step1_items)

# Danh sách các từ khóa liên quan đến "milk4mom" (sữa cho mẹ)
milk4mom_keywords = [
    "milk for mom", "sữa cho mẹ", "sữa mẹ", "milk for breastfeeding", 
    "breast milk", "sữa bầu", "sữa dành cho mẹ", "nước uống cho mẹ"
]

# Tạo biểu thức lọc với các từ khóa (chuyển tất cả chuỗi thành chữ thường trước khi so sánh)
milk4mom_items = item_df.filter(
    pl.col("description_merge").str.to_lowercase().str.contains("|".join(milk4mom_keywords).lower())  # Chuyển thành chữ thường
)["item_id"]

# In kết quả
print(milk4mom_items)

# Lọc giá trị "0M" hoặc "1-3M" trong cột "age_bucket_final"
age_group_items = item_df.filter(pl.col("age_bucket_final").is_in(["0M", "1-3M"]))["item_id"]

print(age_group_items)

shape: (17,)
Series: 'item_id' [str]
[
	"1727000000001"
	"0006040000428"
	"2798000000001"
	"4697000000002"
	"6497000000002"
	…
	"6497000000017"
	"6497000000018"
	"5949000000025"
	"6679000000002"
	"6678000000004"
]
shape: (480,)
Series: 'item_id' [str]
[
	"0020010000438"
	"0020010000440"
	"0020020000051"
	"0006040000143"
	"0020010000289"
	…
	"4666000000004"
	"4667000000003"
	"6597000000001"
	"5194000000001"
	"5194000000002"
]
shape: (2_000,)
Series: 'item_id' [str]
[
	"0502020000004"
	"0020010000151"
	"0007040040001"
	"0007051040005"
	"0014570000020"
	…
	"4684000000003"
	"3389000000006"
	"3524000000153"
	"0502021160016"
	"6996000000174"
]


In [81]:
# 2️⃣ Nhóm theo từng điều kiện
# Sử dụng inner join để chỉ giữ lại customer_id có mặt trong purchase_df
step1_df = (
    purchase_df.filter(pl.col("item_id").is_in(step1_items))  # Lọc từ purchase_df theo item_id
    .group_by("customer_id")
    .agg(pl.col("created_date").min().alias("first_date_buy_step1"))
)
print(step1_df)

mom_df = (
    purchase_df.filter(pl.col("item_id").is_in(milk4mom_items))  # Lọc từ purchase_df theo item_id
    .group_by("customer_id")
    .agg(pl.col("created_date").max().alias("last_date_buy_milk4mom"))
)
print(mom_df)

age_group_df = (
    purchase_df.filter(pl.col("item_id").is_in(age_group_items))  # Lọc từ purchase_df theo item_id
    .group_by("customer_id")
    .agg(pl.col("created_date").min().alias("first_date_buy_age_group_0_3M"))
)
print(age_group_df)


/tmp/ipykernel_2199790/1299262212.py:4: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  purchase_df.filter(pl.col("item_id").is_in(step1_items))  # Lọc từ purchase_df theo item_id


shape: (168_074, 2)
┌─────────────┬─────────────────────────┐
│ customer_id ┆ first_date_buy_step1    │
│ ---         ┆ ---                     │
│ i32         ┆ datetime[μs]            │
╞═════════════╪═════════════════════════╡
│ 3536537     ┆ 2024-10-19 13:58:12.873 │
│ 7581139     ┆ 2024-08-18 20:30:05.923 │
│ 505052      ┆ 2024-12-26 14:27:25.957 │
│ 3109085     ┆ 2024-03-11 14:28:00.370 │
│ 7817345     ┆ 2024-08-27 11:57:03.967 │
│ …           ┆ …                       │
│ 7062716     ┆ 2024-02-26 15:22:42.290 │
│ 6628056     ┆ 2024-03-03 18:25:23.990 │
│ 3973162     ┆ 2024-06-22 17:54:33.690 │
│ 6029020     ┆ 2024-02-03 18:06:17.340 │
│ 7760062     ┆ 2024-08-08 10:59:03.260 │
└─────────────┴─────────────────────────┘


/tmp/ipykernel_2199790/1299262212.py:11: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  purchase_df.filter(pl.col("item_id").is_in(milk4mom_items))  # Lọc từ purchase_df theo item_id


shape: (995_566, 2)
┌─────────────┬─────────────────────────┐
│ customer_id ┆ last_date_buy_milk4mom  │
│ ---         ┆ ---                     │
│ i32         ┆ datetime[μs]            │
╞═════════════╪═════════════════════════╡
│ 1740742     ┆ 2024-11-23 20:16:26.663 │
│ 7890226     ┆ 2024-12-11 21:28:29.117 │
│ 6174127     ┆ 2024-08-26 15:22:44.280 │
│ 7178461     ┆ 2024-12-25 09:37:18.543 │
│ 4687681     ┆ 2024-11-04 19:26:35.687 │
│ …           ┆ …                       │
│ 7411782     ┆ 2024-09-06 18:06:36.273 │
│ 1235821     ┆ 2024-12-12 16:14:01.127 │
│ 7657164     ┆ 2024-07-11 15:58:45.993 │
│ 4901733     ┆ 2024-09-30 19:07:46.330 │
│ 6701854     ┆ 2024-07-09 20:50:48.590 │
└─────────────┴─────────────────────────┘


/tmp/ipykernel_2199790/1299262212.py:18: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  purchase_df.filter(pl.col("item_id").is_in(age_group_items))  # Lọc từ purchase_df theo item_id


shape: (1_162_810, 2)
┌─────────────┬───────────────────────────────┐
│ customer_id ┆ first_date_buy_age_group_0_3M │
│ ---         ┆ ---                           │
│ i32         ┆ datetime[μs]                  │
╞═════════════╪═══════════════════════════════╡
│ 8205211     ┆ 2024-12-31 09:04:54.790       │
│ 7083560     ┆ 2024-04-29 08:49:55.247       │
│ 7114101     ┆ 2024-01-05 15:01:10.663       │
│ 3821243     ┆ 2024-11-07 13:36:53.307       │
│ 6853121     ┆ 2024-01-27 16:59:25.753       │
│ …           ┆ …                             │
│ 7607616     ┆ 2024-06-16 18:27:58.337       │
│ 8106112     ┆ 2024-11-26 10:37:12.080       │
│ 6937798     ┆ 2024-10-13 09:24:12.570       │
│ 4219199     ┆ 2024-01-07 09:52:22.373       │
│ 7150278     ┆ 2024-01-20 10:29:43.203       │
└─────────────┴───────────────────────────────┘


In [96]:
import polars as pl
from datetime import datetime

# 1️⃣ Join từng bước và xóa cột dư
pred_df = step1_df.join(age_group_df, on="customer_id", how="left")  # Left join để giữ tất cả customer_id từ purchase_df

# Nếu join tạo ra customer_id_right → drop
if "customer_id_right" in pred_df.columns:
    pred_df = pred_df.drop("customer_id_right")

# Join với bảng mom_df
pred_df = pred_df.join(mom_df, on="customer_id", how="left")  # Left join để giữ tất cả customer_id từ purchase_df

# Nếu join tạo ra customer_id_right → drop
if "customer_id_right" in pred_df.columns:
    pred_df = pred_df.drop("customer_id_right")

# 2️⃣ Tính toán ngày sinh dựa trên các cột (ngày đầu tiên mua sản phẩm hoặc ngày cuối mua sữa bầu)
pred_df = pred_df.with_columns([
    # age-group: ngày đầu tiên mua sản phẩm (ngày sinh)
    pl.when(pl.col("first_date_buy_age_group_0_3M").is_null())
    .then(pl.lit("1900-01-01"))
    .otherwise(pl.col("first_date_buy_age_group_0_3M"))
    .alias("birth_date_age_group"),

    # step 1: ngày đầu tiên mua sản phẩm cho trẻ sơ sinh (ngày sinh)
    pl.when(pl.col("first_date_buy_step1").is_null())
    .then(pl.lit("1900-01-01"))
    .otherwise(pl.col("first_date_buy_step1"))
    .alias("birth_date_step1"),

    # milk4mom: cộng thêm 1 tháng sau lần cuối mua sữa cho mẹ (ngày sinh giả định)
    pl.when(pl.col("last_date_buy_milk4mom").is_null())
    .then(pl.lit("1900-01-01"))
    .otherwise(
        pl.col("last_date_buy_milk4mom")
        .cast(pl.Date)  # Chuyển đổi thành kiểu Date (nếu cần)
        + pl.duration(days=30)  # Cộng thêm 30 ngày (1 tháng)
    )
    .alias("adjusted_milk4mom")  # Cộng thêm 1 tháng vào ngày mua sữa bầu
])


# 3️⃣ Tính độ tuổi (đơn vị: tháng)
# Để tính độ tuổi theo tháng, ta tính sự chênh lệch giữa ngày hiện tại và ngày sinh, rồi chia cho 30
# Lấy ngày hiện tại và chuyển đổi sang kiểu Datetime
current_date = pl.lit(datetime.now().strftime("%Y-%m-%d")).str.strptime(pl.Datetime, "%Y-%m-%d")

pred_df = pred_df.with_columns([
    # Tính tuổi theo step 1 từ ngày sinh (birth_date_step1)
    ((pl.col("birth_date_step1")
      .str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S.%f", strict=False) - current_date)
      .dt.total_days() / 30).alias("age_by_step1_months"),

    # Tính tuổi theo age-group từ ngày sinh (birth_date_age_group)
    ((pl.col("birth_date_age_group")
      .str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S.%f", strict=False) - current_date)
      .dt.total_days() / 30).alias("age_by_age_group_months"),

    # Tính tuổi cho milk4mom (giả sử sinh một tháng sau ngày cuối mua sữa)
    ((pl.col("adjusted_milk4mom")
      .str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S.%f", strict=False) - current_date)
      .dt.total_days() / 30).alias("age_by_milk4mom_months")
])



# 4️⃣ Chọn cột và tạo bảng theo đúng định dạng yêu cầu
final_df = pred_df.select([
    "customer_id",
    "birth_date_step1",
    "age_by_step1_months",
    "birth_date_age_group",
    "age_by_age_group_months",
    "adjusted_milk4mom",
    "age_by_milk4mom_months"
])

# 5️⃣ Kết quả
print(final_df)


shape: (168_074, 7)
┌─────────────┬──────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┐
│ customer_id ┆ birth_date_s ┆ age_by_step ┆ birth_date_ ┆ age_by_age_ ┆ adjusted_mi ┆ age_by_milk │
│ ---         ┆ tep1         ┆ 1_months    ┆ age_group   ┆ group_month ┆ lk4mom      ┆ 4mom_months │
│ i32         ┆ ---          ┆ ---         ┆ ---         ┆ s           ┆ ---         ┆ ---         │
│             ┆ str          ┆ f64         ┆ str         ┆ ---         ┆ str         ┆ f64         │
│             ┆              ┆             ┆             ┆ f64         ┆             ┆             │
╞═════════════╪══════════════╪═════════════╪═════════════╪═════════════╪═════════════╪═════════════╡
│ 3536537     ┆ 2024-10-19   ┆ -12.0       ┆ 2024-02-03  ┆ -20.633333  ┆ 2024-12-28  ┆ null        │
│             ┆ 13:58:12.873 ┆             ┆ 17:08:22.65 ┆             ┆             ┆             │
│             ┆ 000          ┆             ┆ 0000        ┆             

/tmp/ipykernel_2199790/1257734920.py:53: ChronoFormatWarning: Detected the pattern `.%f` in the chrono format string. This pattern should not be used to parse values after a decimal point. Use `%.f` instead. See the full specification: https://docs.rs/chrono/latest/chrono/format/strftime
  .str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S.%f", strict=False) - current_date)
/tmp/ipykernel_2199790/1257734920.py:58: ChronoFormatWarning: Detected the pattern `.%f` in the chrono format string. This pattern should not be used to parse values after a decimal point. Use `%.f` instead. See the full specification: https://docs.rs/chrono/latest/chrono/format/strftime
  .str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S.%f", strict=False) - current_date)
/tmp/ipykernel_2199790/1257734920.py:63: ChronoFormatWarning: Detected the pattern `.%f` in the chrono format string. This pattern should not be used to parse values after a decimal point. Use `%.f` instead. See the full specification: https://docs.rs/chro

Kiểm tra một số thứ

In [101]:
# Tính số lượng null và tỷ lệ phần trăm của null trong các cột age_by_step1, age_by_age_group, age_by_milk4mom
null_counts = pred_df.select([
    (pl.col("age_by_step1_months").is_null().sum().alias("null_age_by_step1")),
    (pl.col("age_by_age_group_months").is_null().sum().alias("null_age_by_age_group")),
    (pl.col("age_by_milk4mom_months").is_null().sum().alias("null_age_by_milk4mom")),
])

# Tính tổng số dòng dữ liệu
total_count = pred_df.height

# In ra số lượng null và tỷ lệ phần trăm
null_counts = null_counts  # Chuyển sang Pandas để dễ dàng tính toán tỷ lệ phần trăm

for col in null_counts.columns:
    null_count = null_counts[col][0]
    percentage = (null_count / total_count) * 100
    print(f"{col}: Null Count = {null_count}, Percentage = {percentage:.2f}%")


null_age_by_step1: Null Count = 0, Percentage = 0.00%
null_age_by_age_group: Null Count = 12787, Percentage = 7.61%
null_age_by_milk4mom: Null Count = 168074, Percentage = 100.00%
